<a href="https://colab.research.google.com/github/aryaeva16-svg/Team-Quiz-Game/blob/main/Copy_of_Untitled9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving shipment.csv to shipment.csv


In [ ]:
import pandas as pd
import numpy as np

# DataCo files are notoriously not UTF-8 — this avoids the classic UnicodeDecodeError
df = pd.read_csv('/content/shipment.csv', encoding='latin1')

print(df.shape)
print(df.columns.tolist())
df.head()


(180519, 53)
['ï»¿Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shippin

,ï»¿Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,02-03-2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [ ]:
import pandas as pd

keep_cols = [
    'Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
    'Benefit per order', 'Sales per customer', 'Delivery Status',
    'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country',
    'Customer Segment', 'Customer State', 'Department Name', 'Market',
    'Order City', 'Order Country', 'order date (DateOrders)',
    'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price',
    'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total',
    'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
    'Product Name', 'shipping date (DateOrders)', 'Shipping Mode'
]

df = pd.read_csv('/content/shipment.csv', encoding='utf-8-sig', usecols=keep_cols)
print(df.shape)
df.head()
df = pd.read_csv('/content/shipment.csv', encoding='latin1', usecols=lambda c: c.strip('ï»¿') in keep_cols)
df.columns = df.columns.str.strip('ï»¿')
print(df.shape)

(180519, 31)
(180519, 31)


In [ ]:
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('(', '', regex=False)
              .str.replace(')', '', regex=False))

print(df.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_name', 'customer_city', 'customer_country', 'customer_segment', 'customer_state', 'department_name', 'market', 'order_city', 'order_country', 'order_date_dateorders', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_name', 'shipping_date_dateorders', 'shipping_mode']


In [ ]:
df['order_date_dateorders'] = pd.to_datetime(df['order_date_dateorders'], errors='coerce')
df['shipping_date_dateorders'] = pd.to_datetime(df['shipping_date_dateorders'], errors='coerce')

before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

Dropped 0 duplicate rows


In [ ]:
print("Missing values before cleanup:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill numeric NaNs with median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median(numeric_only=True))

# Drop rows with no date (can't be used in time-based analysis anyway)
df = df.dropna(subset=['order_date_dateorders', 'shipping_date_dateorders'])

# Sanity filters — remove impossible values
before = len(df)
df = df[df['order_item_quantity'] > 0]
df = df[df['order_item_product_price'] >= 0]
df = df[df['sales'] >= 0]
print(f"Dropped {before - len(df)} rows with invalid quantity/price/sales")

df.reset_index(drop=True, inplace=True)
print(df.shape)

Missing values before cleanup:
order_date_dateorders        71103
shipping_date_dateorders    108963
dtype: int64
Dropped 0 rows with invalid quantity/price/sales
(20957, 31)


In [ ]:
df['route'] = df['order_country'] + ' -> ' + df['customer_country']
df['delivery_delay_days'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
df['estimated_cost'] = df['sales'] - df['order_profit_per_order']
df['cost_per_unit'] = df['estimated_cost'] / df['order_item_quantity']

HOLDING_COST_PER_UNIT_PER_DAY = 0.75
df['storage_cost'] = (df['days_for_shipment_scheduled'] *
                       HOLDING_COST_PER_UNIT_PER_DAY *
                       df['order_item_quantity'])

df[['route', 'shipping_mode', 'estimated_cost', 'cost_per_unit', 'storage_cost', 'delivery_delay_days']].head()

,route,shipping_mode,estimated_cost,cost_per_unit,storage_cost,delivery_delay_days
0,Indonesia -> Puerto Rico,Standard Class,236.500000,236.500000,3.0,-1
1,Australia -> Puerto Rico,Second Class,66.400002,33.200001,3.0,4
2,TurquÃ­a -> Puerto Rico,Second Class,55.290003,27.645001,3.0,0
3,Mongolia -> Puerto Rico,Second Class,90.900000,45.450000,3.0,1
4,TurquÃ­a -> Puerto Rico,Second Class,121.750000,60.875000,3.0,3


In [ ]:
from google.colab import files

df.to_csv('/content/shipment_cleaned.csv', index=False)
print(f"Final shape: {df.shape}")

import os
size_mb = os.path.getsize('/content/shipment_cleaned.csv') / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")

files.download('/content/shipment_cleaned.csv')

Final shape: (20957, 36)
File size: 7.81 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

features = ['order_item_quantity', 'order_item_discount_rate', 'days_for_shipping_real',
            'days_for_shipment_scheduled', 'delivery_delay_days', 'order_item_product_price']
X = df[features].fillna(0)
y = df['estimated_cost']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(importance)

# Which variables explain ~80% of variance
cumulative = importance.cumsum() / importance.sum()
print(cumulative)

order_item_product_price       0.709056
order_item_discount_rate       0.113031
order_item_quantity            0.084528
delivery_delay_days            0.041154
days_for_shipping_real         0.037118
days_for_shipment_scheduled    0.015113
dtype: float64
order_item_product_price       0.709056
order_item_discount_rate       0.822087
order_item_quantity            0.906615
delivery_delay_days            0.947770
days_for_shipping_real         0.984887
days_for_shipment_scheduled    1.000000
dtype: float64


In [ ]:
# Step 1: Upload the CSV
from google.colab import files
import pandas as pd
import re

uploaded = files.upload()

# Get uploaded file name
file_name = list(uploaded.keys())[0]

# Step 2: Read the CSV
df = pd.read_csv(file_name)

# Step 3: Function to fix encoding issues
def clean_text(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Fix mojibake (UTF-8 read as Latin1)
    try:
        text = text.encode('latin1').decode('utf-8')
    except:
        pass

    # Remove replacement characters
    text = text.replace("�", "")

    # Remove hidden control characters
    text = re.sub(r'[\x00-\x1F\x7F]', '', text)

    return text.strip()
replacements = {
    "MÃƒÂ©xico": "México",
    "EspaÃƒÂ±a": "España",
    "MÃƒÂ©xico ": "México",
    "EspaÃƒÂ±a ": "España"
}

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace(replacements, regex=True)

# Apply cleaning to every text column
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(clean_text)

# Step 4: Save cleaned file
output_file = "cleaned_supply_chain_fixed.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("✅ Cleaning complete!")

# Step 5: Download automatically
files.download(output_file)

Saving query_1.csv to query_1 (1).csv
✅ Cleaning complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>